In [1]:
import requests
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from sqlalchemy import create_engine
import time
import cloudscraper
import re

In [2]:
list_am_main = 'https://www.list.am'
list_am_url = f"{list_am_main}/category/1472?n=0&bid=49&mid=957%2C3284&_a2_1=2012&_a2_2=2017&_a27=0&_a22=0&_a102=0"

In [3]:
list_am_url

'https://www.list.am/category/1472?n=0&bid=49&mid=957%2C3284&_a2_1=2012&_a2_2=2017&_a27=0&_a22=0&_a102=0'

In [27]:
def get_html_response(url):
    scraper = cloudscraper.create_scraper()
    response = scraper.get(url)
    response_text = response.text
    soup = BeautifulSoup(response_text, "html.parser")
    return soup


def get_cars(soup):
    search_results = soup.find_all('div' ,attrs={'class':'dl'})
    return search_results

def get_hrefs(cars):
    if len(cars)>1:
        top_links = cars[0].find('div' ,attrs={'class':'gl'})
        hrefs =  [list_am_main + link['href'] for link in top_links.find_all('a',attrs={'class':"h"})]
        normal_links = cars[1].find('div' ,attrs={'class':'gl'})
        hrefs1 = [list_am_main + link['href'] for link in normal_links.find_all('a',attrs={'class':"class"})]
        hrefs = hrefs + hrefs1
    else:
        hrefs = [list_am_main + link['href'] for link in cars.find_all('a',attrs={'class':"class"})]
    return hrefs

def get_specific_car_content(href):
    scraper = cloudscraper.create_scraper()
    html = scraper.get(href).text
    #html = requests.get(href).text
    soup = BeautifulSoup(html, "html.parser")
    return soup

def get_car_name(car_soup):
    return car_soup.find('title').text.split(' - ')[0]

def get_car_vin(car_soup):
    try:
        return car_soup.find('div',attrs={'class':"pad-left-6"}).text.strip()
    except:
        return

def get_car_year(car_soup):
    return car_soup.find('a',attrs={'class':"grey-text"}).text

def get_car_metadata(car_soup):
    data_dict = {}
    all_details = car_soup.find('div', class_='vi')

    car_detils = all_details.find_all('div',class_='attr g new')
    for detail in car_detils:
        sub_details = detail.find_all('div',class_='at2')
        for sub_detail in sub_details:
            names = sub_detail.find_all('div')
            for name in names[1:]:
                titles = name.find_all('p')
                if len(titles) > 1:
                    tag_element,name_element = titles
                else:
                    tag_element =  titles[0]
                    name_element = titles[0]
                data_dict[name_element.text] = tag_element.text
                data_dict = reverse_keys(data_dict , new_keys=('Հնարավոր է փոխանակում','Մաքսազերծված է'))
                    
    return data_dict


def get_add_info(car_soup):
    add_infos = car_soup.find_all('div',attrs={'class':"nottii bltitle medium"})
    count = 0
    add_exists = False
    for i in add_infos:
        if i.text == 'Լրացուցիչ':
            add_exists = True
            break
        else:
            count += 1
    if add_exists:
        add_info = car_soup.find_all('div',attrs={'class':"ad-options"})[count].text.strip()
    else:
        add_info = None
    return add_info

def get_car_seller_phone(car_soup):
    try:
        return car_soup.find('a' , attrs={'id':"callBtnOptional1"}).text.strip()
    except:
        return car_soup.find('a' , attrs={'id':"callBtn1"}).text.strip()

def get_seller_id(car_soup):
    return car_soup.find('a',class_ = 'n')['href'].split('/')[-1]

def get_car_price(car_soup):
    
    while True:
        element = car_soup.find('span', class_='price x')
        if element:
            break
        else:
            time.sleep(2)
            print('Trying find price')

    return element.text.strip()

def get_add_info(car_soup):

    def extract_post_id(span):
        return span.text.split(' ')[-1]
        return re.search(r'\d+', span.text).group()

    def extract_create_date(span):
        return span.text.split(' ')[-1]

    def extract_update_date(span):
        if span: 
            return ' '.join(span.text.split(' ')[-2:]) 
    
    description = car_soup.find('div', class_='vi').find('div', class_='body').text
    other_info = car_soup.find('div', class_='vi').find('div', class_='footer').find_all('span')
    
    if len(other_info) == 3:
        post_id_span,create_date_span,update_date_span = other_info
    else:
        post_id_span,create_date_span = other_info
        update_date_span = None
    post_id,create_date,update_date = extract_post_id(post_id_span) ,extract_create_date(create_date_span),extract_update_date(update_date_span)
    return description,post_id,create_date,update_date

def reverse_keys(metadata,new_keys):
    new_dict = {}
    for i in metadata.items():
        for j in new_keys:
            if j == i[1]:
                new_dict[j] = i[0]
    
    for value in new_dict.values():
        del metadata[value]
    metadata.update(new_dict)
    return metadata


In [28]:
soup = get_html_response(list_am_url)
cars = get_cars(soup)
hrefs = get_hrefs(cars)

In [29]:
print(f'Found {len(hrefs)} cars')

Found 30 cars


In [32]:
count = 0
cars_list = []
for href in hrefs: #filter(lambda x: x == 'https://www.list.am/item/24081985?ld_src=2',hrefs):
    print(href)
    car_soup = get_specific_car_content(href)
    car_name = get_car_name(car_soup)
    price = get_car_price(car_soup)
    metadata = get_car_metadata(car_soup=car_soup)
    #metadata = reverse_keys(metadata , new_keys=('Հնարավոր է փոխանակում','Մաքսազերծված է'))
    description,post_id,create_date,update_date = get_add_info(car_soup)
    seller_id = get_seller_id(car_soup)
    metadata['car_name']=car_name
    metadata['price'] = price
    metadata['description'] = description
    metadata['post_id'] = post_id
    metadata['create_date'] = create_date
    metadata['update_date'] = update_date
    metadata['Link'] = href
    cars_list.append(metadata)

https://www.list.am/item/24081985?ld_src=2
https://www.list.am/item/24078167?ld_src=2
https://www.list.am/item/24084413?ld_src=2
https://www.list.am/item/23587943?ld_src=2
https://www.list.am/item/23555146?ld_src=2
https://www.list.am/item/23831645?ld_src=2
https://www.list.am/item/20848095?ld_src=2
https://www.list.am/item/23828244?ld_src=2
https://www.list.am/item/23446961?ld_src=2
https://www.list.am/item/23360331?ld_src=2
https://www.list.am/item/23820362?ld_src=2
https://www.list.am/item/21988065?ld_src=2
https://www.list.am/item/24104502?ld_src=2
https://www.list.am/item/23991920?ld_src=2
https://www.list.am/item/20844810?ld_src=2
https://www.list.am/item/23236278?ld_src=2
https://www.list.am/item/22620693?ld_src=2
https://www.list.am/item/24029867?ld_src=2
https://www.list.am/item/24051460?ld_src=2
https://www.list.am/item/23795935?ld_src=2
https://www.list.am/item/24097155?ld_src=2
https://www.list.am/item/23964486?ld_src=2
https://www.list.am/item/24005250?ld_src=2
https://www

In [33]:
df = pd.DataFrame(cars_list)

In [34]:
df['scrape_date'] = pd.to_datetime(datetime.today())

In [36]:


# 2. Define your PostgreSQL credentials
db_user = 'postgres'
db_password = 'postgres'
db_host = 'localhost'       # Use your server IP if it is not hosted locally
db_port = '5432'            # 5432 is the default PostgreSQL port
db_name = 'car_db'

# 3. Create the SQLAlchemy engine for PostgreSQL
connection_string = f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}'
engine = create_engine(connection_string)


In [ ]:

# 4. Append the data to the table
df.to_sql(
    name='list_am_listings', 
    con=engine, 
    if_exists='append', 
    index=False
)

print(f"Successfully appended {len(df)} rows to PostgreSQL for {df['scrape_date'].iloc[0]}")

Successfully added 30 rows to PostgreSQL for 2026-08-16 16:34:23.408414
